# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
# TODO: Load environment variables
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# retrieve_game tool
import chromadb
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

# Connect to the persisted ChromaDB and load the existing collection
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)


@tool
def retrieve_game(query: str) -> list:
    """Semantic search: Finds the most relevant games in the vector DB.

    args:
    - query: a question about the game industry.

    You'll receive results as a list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(
        query_texts=[query],
        n_results=5,
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    games = []
    for doc, meta in zip(documents, metadatas):
        games.append({
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description"),
            "Content": doc,
        })
    return games


#### Evaluate Retrieval Tool

In [5]:
# evaluate_retrieval tool (LLM as judge)
from typing import List
from pydantic import BaseModel, Field
from lib.llm import LLM


class EvaluationReport(BaseModel):
    """Structured judgment about whether retrieved docs can answer the question."""
    useful: bool = Field(
        description="Whether the retrieved documents are enough to answer the question"
    )
    description: str = Field(
        description="Detailed explanation justifying the decision"
    )


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[dict]) -> dict:
    """Based on the user's question and on the list of retrieved documents,
    it analyzes the usability of the documents to respond to that question.

    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini", temperature=0.0)

    system_prompt = (
        "You are an evaluator of a RAG system for the video game industry. "
        "Your task is to evaluate if the documents are enough to respond to the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not."
    )
    user_prompt = (
        f"Question: {question}\n\n"
        f"Retrieved documents:\n{retrieved_docs}"
    )

    ai_message = llm.invoke(
        [SystemMessage(content=system_prompt), UserMessage(content=user_prompt)],
        response_format=EvaluationReport,
    )

    report = EvaluationReport.model_validate_json(ai_message.content)
    print(f"Evaluation result: {report}")
    return report.model_dump()


#### Game Web Search Tool

In [6]:
# game_web_search tool (Tavily) — returns sources so the agent can cite them
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


@tool
def game_web_search(question: str) -> dict:
    """Web search: Looks up information about the game industry on the web.
    Use this when the internal vector DB does not contain enough information
    to answer the question.

    args:
    - question: a question about the game industry.

    Returns a dict with:
    - answer: a short synthesized answer from the web (may be None)
    - sources: a list of {title, url, content} to be used as citations
    """
    response = tavily_client.search(
        query=question,
        max_results=5,
        include_answer=True,
    )

    sources = [
        {
            "title": r.get("title"),
            "url": r.get("url"),
            "content": r.get("content"),
        }
        for r in response.get("results", [])
    ]

    return {
        "answer": response.get("answer"),
        "sources": sources,
    }


### Agent

The agent is implemented as a **class**, `UdaPlayAgent`, defined in the next cell.
It subclasses the base `Agent` (`lib/agents.py`), so it is a reusable, stateful
object: it owns its tools and system instructions, and it keeps a
`ShortTermMemory` that persists conversation state per `session_id`. Each call to
`agent.invoke(query, session_id=...)` replays the prior turns of that session, so
the agent remembers previous context across multiple queries.

Internally, the agent's workflow is a **state machine** (prepare messages → call
the LLM → run any tools → loop back → terminate), so every `invoke` advances a
well-defined set of nodes rather than ad-hoc procedural code.

In [11]:
# Implement the UdaPlay agent as a CLASS.
#
# UdaPlayAgent subclasses the base `Agent` (lib/agents.py), so it inherits a
# stateful, production-ready design: its workflow runs as a state machine and it
# remembers conversation history per `session_id` via ShortTermMemory. The class
# bundles its own system instructions and tools, so callers just do
# `UdaPlayAgent().invoke(question, session_id=...)`.
from lib.agents import Agent


class UdaPlayAgent(Agent):
    """An AI research agent for the video game industry.

    Implemented as a class so the agent is a reusable, stateful object:
    it owns its instructions and tools, executes its workflow as a state
    machine, and maintains conversation state across multiple queries
    within the same session_id.
    """

    INSTRUCTIONS = (
        "You are UdaPlay, an AI research assistant for the video game industry. "
        "You answer questions about video games such as release years, platforms, "
        "genres and publishers.\n\n"
        "Follow this strategy for every question:\n"
        "1. Call `retrieve_game` to search the internal vector database first.\n"
        "2. Call `evaluate_retrieval` with the user's question and the retrieved "
        "documents to decide whether they are enough to answer.\n"
        "3. If the retrieval is NOT useful, call `game_web_search` to look the answer "
        "up on the web.\n"
        "4. Give a clear, concise final answer. Include the platform and release year "
        "when relevant.\n\n"
        "Citations are REQUIRED. End every answer with a 'Sources:' section:\n"
        "- If you answered from the internal database, write 'Sources: internal game database'.\n"
        "- If you used `game_web_search`, list the specific source URLs you relied on "
        "(from the 'sources' field returned by the tool), one per line.\n\n"
        "Never invent facts or URLs. If you cannot find the answer, say so."
    )

    def __init__(self, model_name: str = "gpt-4o-mini", temperature: float = 0.0):
        super().__init__(
            model_name=model_name,
            instructions=self.INSTRUCTIONS,
            tools=[retrieve_game, evaluate_retrieval, game_web_search],
            temperature=temperature,
        )


# Instantiate the agent
agent = UdaPlayAgent()

In [12]:
# Invoke the agent across a SINGLE session so every query shares conversation
# state. This proves the rubric points in one shot:
#   1. The Agent (a stateful class) handles multiple queries within one session.
#   2. It remembers previous context: Q3 below never names the game, so the agent
#      can only answer "it" by recalling what it said in Q2.
SESSION_ID = "udaplay-session"

questions = [
    "When was Pokemon Gold and Silver released?",       # internal vector DB
    "Which one was the first 3D platformer Mario game?",   # internal vector DB
    "And which console was it released on?",               # relies on memory of Q2
    "Was Mortal Kombat X released for PlayStation 5?",     # falls back to web search
]


def print_tool_usage(messages):
    """Print which tools the agent called (with args) and a preview of each result."""
    for msg in messages:
        # Tool calls requested by the assistant
        tool_calls = getattr(msg, "tool_calls", None)
        if getattr(msg, "role", None) == "assistant" and tool_calls:
            for call in tool_calls:
                print(f"   [tool call]   {call.function.name}({call.function.arguments})")
        # Results returned by the tools
        if getattr(msg, "role", None) == "tool":
            preview = (msg.content or "").replace("\n", " ")[:200]
            print(f"   [tool result] {msg.name}: {preview}...")

prev_len = 0
for question in questions:
    run = agent.invoke(question, session_id=SESSION_ID)
    messages = run.get_final_state()["messages"]
    new_messages = messages[prev_len:]
    prev_len = len(messages)
    answer = messages[-1].content

    print(f"Q: {question}")
    print("Tool usage:")
    print_tool_usage(new_messages)
    print(f"\nA: {answer}")
    print("-" * 80)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Evaluation result: useful=True description='The retrieved document provides clear information regarding the release year of Pokemon Gold and Silver, stating that they were released in 1999. Additionally, it includes relevant details about the platform (Game Boy Color) and a brief description of the games, which further contextualizes the answer. Therefore, the document is sufficient to respond to the query about the release date.'
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q: When was Pokemon Gold and Silver released?
Tool usage:
   [tool call]   retrieve_game({"query":"Pokemon Gold and Silver release date"})
   [tool result] retrieve_game: "[{'Platform': 'Game Boy Color', 'Name': 'Pok\

### Session Memory (proven above, not re-tested)

All four questions above ran inside a single `session_id` (`"udaplay-session"`),
so the agent treats them as one ongoing conversation. The third question — *"And
which console was it released on?"* — never names the game; the agent can only
answer it because the `Agent` class persists each turn in `ShortTermMemory` keyed
by `session_id` and replays that history on the next `invoke`.

The cell below simply *inspects* that session to confirm every turn was stored in
the same conversation — it does not re-run the queries.

In [13]:
# Confirm the whole conversation lived in ONE shared session (no re-testing).
# get_session_runs returns every Run stored under the session_id, one per query.
runs = agent.get_session_runs(SESSION_ID)
print(f"Session '{SESSION_ID}' holds {len(runs)} stored turns (one per query).")

# The final run still carries the full message history of the entire session,
# which is exactly what lets the agent remember earlier turns.
final_messages = runs[-1].get_final_state()["messages"]
user_turns = [m.content for m in final_messages if getattr(m, "role", None) == "user"]
print(f"\nUser questions remembered in this single session ({len(user_turns)}):")
for q in user_turns:
    print(f"  - {q}")

Session 'udaplay-session' holds 4 stored turns (one per query).

User questions remembered in this single session (4):
  - When was Pokemon Gold and Silver released?
  - Which one was the first 3D platformer Mario game?
  - And which console was it released on?
  - Was Mortal Kombat X released for PlayStation 5?
